# Fase 3: Preservação de Memória via Diferenciação Fracionária (FFD)

Este notebook demonstra o algoritmo **Fixed-Width Window Fractional Differentiation (FFD)** de Marcos López de Prado:
1. **Geração de pesos do operador binomial $(1-B)^d$** com ponto de corte `thres` para largura de janela constante.
2. **Convolução na série de log-preços** para gerar séries estatisticamente estacionárias.
3. **Otimizador de Estacionariedade (`find_optimal_d`)**: Testa valores de $d$ de $0.0$ a $1.0$ e aplica o teste Augmented Dickey-Fuller (ADF) para encontrar o menor $d$ que rejeita a hipótese nula ($p < 0.05$), preservando a máxima memória e correlação com a série original.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Adicionar diretório raiz ao path
sys.path.insert(0, os.path.abspath('..'))

from src.features.fracdiff import (
    get_ffd_weights,
    frac_diff_ffd,
    find_optimal_d,
    plot_min_ffd,
)

## 1. Pesos da Diferenciação Fracionária $(1-B)^d$

In [2]:
for d_val in [0.2, 0.4, 0.6, 0.8, 1.0]:
    w = get_ffd_weights(d=d_val, thres=1e-3)
    print(f"d = {d_val:.1f} -> Número de lags da janela (Threshold 1e-3): {len(w)}")

## 2. Busca do $d$ Ótimo (ADF Test vs Correlação)

In [3]:
np.random.seed(42)
n_obs = 500
dates = pd.date_range("2024-01-01", periods=n_obs, freq="B")
rets = np.random.normal(0.0004, 0.012, size=n_obs)
price_series = pd.Series(100.0 * np.exp(np.cumsum(rets)), index=dates, name="SP500_Simulated")

res_opt = find_optimal_d(price_series, d_range=(0.0, 1.0), step=0.05, p_val_threshold=0.05, thres=1e-4)

print(f"=== RESULTADO DA OTIMIZAÇÃO FFD ===")
print(f"d Ótimo: {res_opt['optimal_d']}")
print(f"p-valor ADF: {res_opt['optimal_p_value']:.4f}")
print(f"Correlação Mantida: {res_opt['optimal_correlation']:.4f}")

plot_min_ffd(res_opt['results_df'])
plt.show()